# Multi-Agent Workflow Evaluation with EvalAI

This notebook shows how to **trace and evaluate multi-agent systems** using the EvalAI Python SDK.

You'll learn:
1. Trace agent workflows with spans, handoffs, and decisions
2. Track costs across agents
3. Evaluate agent outputs with assertions
4. Use the `@WithContext` decorator for context propagation

In [ ]:
# Install (run once)
# !pip install pauly4010-evalai-sdk

## 1. Simulated Multi-Agent System

We'll create a research pipeline with three agents: **Researcher**, **Analyst**, and **Writer**.

In [ ]:
import asyncio


async def researcher(query: str) -> dict:
    """Simulates a research agent that gathers information."""
    await asyncio.sleep(0.1)  # simulate API call
    return {
        "findings": f"Research findings for '{query}': AI adoption grew 35% in 2025. "
        "Key areas include healthcare, finance, and autonomous systems.",
        "sources": ["arxiv.org", "nature.com", "ieee.org"],
        "confidence": 0.87,
    }


async def analyst(findings: dict) -> dict:
    """Simulates an analyst agent that processes research."""
    await asyncio.sleep(0.05)
    return {
        "summary": "AI adoption is accelerating across industries, with healthcare leading.",
        "key_metrics": {"growth_rate": 0.35, "top_sector": "healthcare"},
        "risk_factors": ["regulation", "data privacy", "talent shortage"],
    }


async def writer(analysis: dict) -> str:
    """Simulates a writer agent that produces the final report."""
    await asyncio.sleep(0.05)
    return (
        f"AI Industry Report: {analysis['summary']} "
        f"Growth rate stands at {analysis['key_metrics']['growth_rate']:.0%}, "
        f"with {analysis['key_metrics']['top_sector']} as the leading sector. "
        f"Key risks include {', '.join(analysis['risk_factors'])}."
    )

## 2. Trace the Workflow

Use `WorkflowTracer` to instrument every agent span, handoff, decision, and cost.

> This example uses a mock client. Replace with `AIEvalClient.init()` to send real traces.

In [ ]:
from unittest.mock import AsyncMock, MagicMock

from evalai_sdk import WorkflowTracer
from evalai_sdk.types import (
    CostCategory,
    DecisionType,
    HandoffType,
    RecordCostParams,
    RecordDecisionParams,
)

# Mock client for local demo (replace with AIEvalClient.init() for real tracing)
mock_client = MagicMock()
mock_client.traces = MagicMock()
mock_client.traces.create = AsyncMock(return_value=MagicMock(id=1, trace_id="demo-trace"))
mock_client.traces.create_span = AsyncMock(return_value=MagicMock(id=1))

tracer = WorkflowTracer(mock_client)

# Start the workflow
await tracer.start_workflow("research-pipeline")
print(f"Workflow started: {tracer.is_active}")

# Agent 1: Researcher
span1 = await tracer.start_agent_span("researcher", {"query": "AI trends 2025"})
findings = await researcher("AI trends 2025")
await tracer.end_agent_span(span1, output=findings)
await tracer.record_cost(RecordCostParams(
    agent_name="researcher", category=CostCategory.LLM_INPUT, amount=0.03, tokens=1200,
))
print(f"Researcher done — confidence: {findings['confidence']}")

# Handoff: Researcher -> Analyst
await tracer.record_handoff("researcher", "analyst", handoff_type=HandoffType.DELEGATION)
await tracer.record_decision(RecordDecisionParams(
    agent_name="router",
    decision_type=DecisionType.ROUTING,
    options=["analyst", "fact-checker"],
    chosen="analyst",
    reasoning="Research confidence > 0.8, skip fact-check",
))

# Agent 2: Analyst
span2 = await tracer.start_agent_span("analyst", {"input": "findings"})
analysis = await analyst(findings)
await tracer.end_agent_span(span2, output=analysis)
await tracer.record_cost(RecordCostParams(
    agent_name="analyst", category=CostCategory.LLM_OUTPUT, amount=0.02, tokens=800,
))
print(f"Analyst done — top sector: {analysis['key_metrics']['top_sector']}")

# Handoff: Analyst -> Writer
await tracer.record_handoff("analyst", "writer", handoff_type=HandoffType.DELEGATION)

# Agent 3: Writer
span3 = await tracer.start_agent_span("writer", {"input": "analysis"})
report = await writer(analysis)
await tracer.end_agent_span(span3, output={"report": report})
await tracer.record_cost(RecordCostParams(
    agent_name="writer", category=CostCategory.LLM_OUTPUT, amount=0.04, tokens=2000,
))

# End workflow
await tracer.end_workflow()

print(f"\nWorkflow complete!")
print(f"Total cost: ${tracer.get_total_cost():.2f}")
print(f"Agents: {len(tracer._spans)}")
print(f"Handoffs: {len(tracer._handoffs)}")
print(f"Decisions: {len(tracer._decisions)}")

## 3. Evaluate Agent Outputs

Use assertions to validate each agent's output quality:

In [ ]:
from evalai_sdk import expect, run_assertions

print("── Researcher Output ──")
researcher_checks = run_assertions([
    lambda: expect(findings["findings"]).to_contain_keywords(["AI", "adoption"]),
    lambda: expect(findings["confidence"]).to_be_between(0.0, 1.0),
    lambda: expect(len(findings["sources"])).to_be_greater_than(0),
])
for r in researcher_checks:
    print(f"  [{'PASS' if r.passed else 'FAIL'}] {r.assertion_type}")

print("\n── Analyst Output ──")
analyst_checks = run_assertions([
    lambda: expect(analysis["summary"]).to_have_length(min=20),
    lambda: expect(analysis["key_metrics"]["growth_rate"]).to_be_between(0.0, 1.0),
    lambda: expect(len(analysis["risk_factors"])).to_be_greater_than(0),
])
for r in analyst_checks:
    print(f"  [{'PASS' if r.passed else 'FAIL'}] {r.assertion_type}")

print("\n── Writer Output (Final Report) ──")
writer_checks = run_assertions([
    lambda: expect(report).to_contain_keywords(["AI", "healthcare", "growth"]),
    lambda: expect(report).to_not_contain_pii(),
    lambda: expect(report).to_be_professional(),
    lambda: expect(report).to_have_proper_grammar(),
    lambda: expect(report).to_have_length(min=50, max=1000),
])
for r in writer_checks:
    print(f"  [{'PASS' if r.passed else 'FAIL'}] {r.assertion_type}")

all_checks = researcher_checks + analyst_checks + writer_checks
passed = sum(1 for r in all_checks if r.passed)
print(f"\nTotal: {passed}/{len(all_checks)} checks passed")

## 4. Context Propagation with `@WithContext`

Use the `@WithContext` decorator to automatically propagate metadata through your agent pipeline:

In [ ]:
from evalai_sdk import WithContext, get_current_context


@WithContext({"service": "research-pipeline", "version": "2.0"})
async def run_pipeline(query: str) -> str:
    ctx = get_current_context()
    print(f"Context: {ctx}")

    findings = await researcher(query)
    analysis = await analyst(findings)
    return await writer(analysis)


final_report = await run_pipeline("AI trends 2025")
print(f"\nReport: {final_report[:100]}...")

# Context is automatically cleaned up after the function returns
print(f"Context after: {get_current_context()}")

## 5. End-to-End Test Suite for the Pipeline

In [ ]:
from evalai_sdk import create_test_suite
from evalai_sdk.types import TestSuiteCase, TestSuiteConfig


async def full_pipeline(query: str) -> str:
    """End-to-end pipeline for test suite evaluation."""
    findings = await researcher(query)
    analysis = await analyst(findings)
    return await writer(analysis)


suite = create_test_suite(
    "agent-pipeline-e2e",
    TestSuiteConfig(
        evaluator=full_pipeline,
        test_cases=[
            TestSuiteCase(
                name="ai-trends",
                input="AI trends 2025",
                assertions=[
                    {"type": "contains_keywords", "value": ["AI", "healthcare"]},
                    {"type": "not_contains_pii"},
                    {"type": "has_length", "value": {"min": 50}},
                ],
            ),
            TestSuiteCase(
                name="climate-research",
                input="Climate technology investments",
                assertions=[
                    {"type": "has_length", "value": {"min": 30}},
                    {"type": "not_contains_pii"},
                ],
            ),
        ],
    ),
)

result = await suite.run()

print(f"Suite: {result.suite_name}")
print(f"Passed: {result.passed_count}/{result.total}")
print(f"Score: {result.score:.2f}")
print(f"Duration: {result.duration_ms:.0f}ms\n")

for r in result.results:
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{status}] {r.test_case_name} ({r.duration_ms:.0f}ms)")